# Skin Lesion XAI-Trust -- Colab Training/Evaluation Run

Runs the full pipeline from `skin-lesion-xai-trust`: dataset prep, training (hybrid CNN-Transformer + 3 baselines), and evaluation (classification metrics, multi-method XAI faithfulness, MC-Dropout uncertainty, Trust Score).

**Before running:** Runtime -> Change runtime type -> GPU. You will be asked to upload your own `kaggle.json` API token (from kaggle.com -> Settings -> Create New Token) -- it is used only locally in this Colab session, never sent anywhere else.

All results (`runs/*/best.pt`, `runs/eval_report_*.json`) are copied to Google Drive at the end so they survive session disconnects.

## 1. Clone the repository

In [ ]:
!git clone https://github.com/DivyaKathirvelan98/skin-lesion-xai-trust.git
%cd skin-lesion-xai-trust
!pip install -q -r requirements.txt

## 2. Kaggle authentication (upload your own kaggle.json)

In [ ]:
from google.colab import files
import os

print("Upload your kaggle.json (Kaggle -> Settings -> API -> Create New Token)")
uploaded = files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
for fname in uploaded:
    os.rename(fname, '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

## 3. Download HAM10000 images/metadata + ground-truth segmentation masks

In [ ]:
!bash scripts/download_ham10000_kaggle.sh data/raw

## 4. Lesion-wise (patient-wise) leakage-safe split

In [ ]:
!python -m src.preprocessing.dataset_split --metadata data/raw/HAM10000_metadata.csv --out data/splits

## 5. Update config to point at the segmentation masks (needed for faithfulness metrics)

In [ ]:
import yaml

with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['data']['segmentations_dir'] = 'data/raw/segmentations'
with open('configs/config.yaml', 'w') as f:
    yaml.safe_dump(cfg, f)
print(cfg['data'])

## 6. Train the proposed hybrid model and all three baselines

Reduce `training.epochs` in `configs/config.yaml` first if you need a faster/cheaper run --
the paper should report whatever was actually run, not a target that wasn't executed.

In [ ]:
for model_name in ['hybrid_cnn_transformer', 'resnet50', 'efficientnet_b0', 'vit_base']:
    print(f'=== training {model_name} ===')
    !python -m src.train --config configs/config.yaml --model {model_name}

## 7. Evaluate every model (classification metrics; XAI/uncertainty/Trust Score for the hybrid model)

In [ ]:
for model_name in ['hybrid_cnn_transformer', 'resnet50', 'efficientnet_b0', 'vit_base']:
    print(f'=== evaluating {model_name} ===')
    !python -m src.evaluate --config configs/config.yaml --model {model_name} \
        --checkpoint runs/{model_name}/best.pt --out runs/eval_report_{model_name}.json

## 8. Consolidate results into one table

In [ ]:
import json
import pandas as pd

rows = []
for model_name in ['hybrid_cnn_transformer', 'resnet50', 'efficientnet_b0', 'vit_base']:
    with open(f'runs/eval_report_{model_name}.json') as f:
        report = json.load(f)['report']
    report['model'] = model_name
    rows.append(report)

results_df = pd.DataFrame(rows).set_index('model')
results_df.to_csv('runs/results_summary.csv')
results_df

## 9. Copy results to Google Drive (survives session disconnects)

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')
dest = '/content/drive/MyDrive/skin-lesion-xai-trust-results'
shutil.copytree('runs', dest, dirs_exist_ok=True)
print(f'Copied results to {dest}')